<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-architect-program/blob/main/Day_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Build a 2-Tool Assistant: Calculator + Weather Lookup
## A Hands-On Introduction to Tool Calling with Claude


---
## Section 1: Understanding Tool Selection & Our Two Tools



### The Toolbox Analogy

Imagine Claude is a smart assistant with a toolbox:

```
┌──────────────────────────┐
│   Claude's Toolbox       │
├──────────────────────────┤
│ 📐 Calculator Tool       │
│ 🌤️ Weather Tool          │
└──────────────────────────┘
```

When you ask Claude a question, Claude looks at the question and decides:
- **Which tool(s) do I need?**
- **Should I grab the calculator?**
- **Should I grab the weather tool?**
- **Should I grab both?**
- **Do I need any tool at all?**

Just like a receptionist at a hospital who listens to your symptoms and directs you
to the right department (cardiology for heart problems, orthopedics for broken bones),
Claude reads your question and decides which tool to use.

### Our Two Tools Today

#### Tool 1: Calculator ✖️➕➖
- **What it does:** Takes a math expression like "50 + 25" and returns the answer
- **When Claude uses it:** When you ask anything math-related
- **Example:** "What is 156 + 89?"

#### Tool 2: Weather Lookup 🌡️
- **What it does:** Takes a city name and returns the current weather
- **When Claude uses it:** When you ask about weather in a location
- **Example:** "What's the weather in Mumbai?"

### Quick Thought Exercise 🧠

Look at these prompts. For each one, which tool do you think Claude would choose?




| User Prompt | Tool Used | Why Claude Chooses This |
|---|---|---|
| "What is 156 + 89?" | Calculator | You asked a math question |
| "What's the weather in Mumbai?" | Weather | You asked about a location's weather |
| "Calculate 50 × 12 and tell me if it's raining in Delhi" | Both Tools | You asked TWO different things |
| "Is 2+2 equal to 4?" | None | Claude knows this from general knowledge |
| "What will the weather be like for my 345 km trip to Bangalore?" | Weather | The context is about location & weather |

### The Key Insight

**Claude is smart enough to:**
- Recognize when it needs a tool (and when it doesn't)
- Call one tool, two tools, or no tools in a single response
- Combine information from multiple tools to answer complex questions

This "smartness" comes from you telling Claude: *"Here's my toolbox and what each tool does."*
Then Claude makes the right choice automatically.



### ✅ Checkpoint: Do You Get It?

**Question:** If I ask Claude "What is the square root of 144?", which tool will it choose? Why might it NOT use the calculator?

**Think about it for 10 seconds, then scroll down...**

<details>
<summary>Click to reveal the answer</summary>

Claude will use **no tool** because it already knows that √144 = 12 from its general knowledge.
It doesn't need to run the calculator tool—it can answer directly.

However, Claude *could* choose to use the calculator tool if you ask something like
"Use the calculator to verify that 12² equals 144." In that case, you're explicitly
asking it to use the tool.

The lesson: **Claude is smart enough to know when tools are actually necessary.**

</details>



---
## Section 2: How Claude Chooses Between Tools

### The Three-Step Process

Here's what happens behind the scenes when you send a prompt to Claude:

```
Step 1: Claude reads your prompt + the tool definitions
              ↓
Step 2: Claude decides which tool(s) to use (or none)
              ↓
Step 3: Claude returns a "tool_use" request (not the answer yet!)
```

### Critical Truth: Claude Does NOT Run Tools

This is important: **Claude does NOT actually run your calculator or check real weather data.**

Instead, Claude says:
> "Hey, I think you need the calculator tool. Please run it with the expression '100 / 4'."

Then **your Python code** runs the tool. You get the result. You send it back to Claude.
Claude reads the result and gives the final answer.

**You control execution. Claude only requests.**

### Four Example Scenarios

#### Scenario 1: Claude Uses General Knowledge (No Tool)

**Your Prompt:**
```
User: "Is Mumbai on the Arabian Sea?"
```

**What Claude Thinks:**
- "The user asked about Mumbai's geography"
- "This is general knowledge—no tool needed"
- "I'll answer directly from my training data"

**Claude's Response:**
```
"Yes, Mumbai is located on the western coast of India,
on the Arabian Sea. It's India's largest city and a major
port and financial center."
```

**Tool Used:** ❌ None

---

#### Scenario 2: Claude Requests the Calculator Tool

**Your Prompt:**
```
User: "What is 1250 divided by 50?"
```

**What Claude Thinks:**
- "This is a math problem"
- "The calculator tool is perfect for this"
- "I'll request the calculator tool"

**Claude's Response (raw format):**
```json
{
  "type": "tool_use",
  "id": "toolu_0123456789...",
  "name": "calculator",
  "input": {
    "expression": "1250 / 50"
  }
}
```

**What Happens Next:**
1. Your Python code sees this tool_use request
2. Your code runs: `calculator_tool("1250 / 50")`
3. Your code gets back: `25.0`
4. Your code sends: "Tool result: 25.0" back to Claude
5. Claude reads the result and gives you the final answer:

**Claude's Final Answer:**
```
"1250 divided by 50 equals 25."
```

**Tool Used:** ✖️ Calculator

---

#### Scenario 3: Claude Requests the Weather Tool

**Your Prompt:**
```
User: "Is it sunny in Bangalore right now?"
```

**What Claude Thinks:**
- "User is asking about weather in a specific city"
- "The weather_lookup tool is perfect"
- "I'll request that tool"

**Claude's Response (raw format):**
```json
{
  "type": "tool_use",
  "id": "toolu_0123456789...",
  "name": "weather_lookup",
  "input": {
    "city": "Bangalore"
  }
}
```

**What Happens Next:**
1. Your Python code sees the tool_use request
2. Your code runs: `weather_lookup_tool("Bangalore")`
3. Your code gets back: Mock weather data (sunny, 28°C, 60% humidity, etc.)
4. Your code sends the result back to Claude
5. Claude reads the result and answers:

**Claude's Final Answer:**
```
"Yes, based on current weather data, Bangalore is sunny right now
with a temperature of 28°C and moderate humidity."
```

**Tool Used:** 🌤️ Weather

---

#### Scenario 4: Claude Requests BOTH Tools

**Your Prompt:**
```
User: "Calculate 500 + 200 and check the weather in Chennai"
```

**What Claude Thinks:**
- "User asked for TWO things:"
  - "A math calculation → use calculator"
  - "Weather lookup → use weather tool"
- "I need both tools!"

**Claude's Response (raw format, contains TWO tool_use blocks):**
```json
[
  {
    "type": "tool_use",
    "id": "toolu_calc_789...",
    "name": "calculator",
    "input": {
      "expression": "500 + 200"
    }
  },
  {
    "type": "tool_use",
    "id": "toolu_weather_456...",
    "name": "weather_lookup",
    "input": {
      "city": "Chennai"
    }
  }
]
```

**What Happens Next:**
1. Your Python code detects TWO tool_use blocks
2. Your code runs BOTH tools
3. Your code gets: Calculator result (700) + Weather result (sunny, 32°C)
4. Your code sends both results back to Claude
5. Claude reads both and combines them:

**Claude's Final Answer:**
```
"500 + 200 equals 700. In Chennai, the weather is currently
sunny with a temperature of 32°C and 55% humidity."
```

**Tool Used:** ✖️ + 🌤️ Both Tools

---

#### Scenario 5: Claude Recognizes Limitations (Boundary Case)

**Your Prompt:**
```
User: "What is the population of Kolkata?"
```

**What Claude Thinks:**
- "User asked about current population statistics"
- "I don't have a 'population lookup' tool"
- "My training data might be outdated"
- "I shouldn't guess or misuse the tools I have"

**Claude's Response:**
```
"I don't have a reliable tool to provide you with the current
population of Kolkata. My training data has a knowledge cutoff,
and population figures change over time. For current data, I'd
recommend checking official sources like Census India or recent
city statistics."
```

**Tool Used:** ❌ None (Claude wisely doesn't misuse tools)

---

### ✅ Checkpoint: Do You Understand the Flow?

**Question:** When Claude responds with a tool_use block, who actually runs the tool?

<details>
<summary>Click to reveal the answer</summary>

**Your Python code runs the tool.** Claude never executes tools—it only *requests* them.

The flow is always:
1. Claude says: "Please run this tool with these inputs"
2. Your code says: "Okay, running it now..." (and actually runs it)
3. Your code says: "Here's the result"
4. Claude says: "Thanks! Here's my final answer"

This separation is important because it lets YOU control what tools run and what data is passed around.

</details>



---
## Section 3: Building the Calculator Tool

### Anatomy of a Tool

Every tool has **two parts:**

1. **The Schema** — A JSON description telling Claude:
   - What the tool is called
   - What it does (description)
   - What inputs it needs (parameter names, types, descriptions)

2. **The Function** — Python code that:
   - Takes the inputs Claude specifies
   - Does the actual work
   - Returns a result

Think of it like a recipe:
- **Schema = The menu description** ("Our cookies are chocolate chip, homemade daily")
- **Function = The actual cooking** (mix flour, eggs, sugar, bake at 350°F)

### The Calculator Tool Schema

Here's what we'll tell Claude about our calculator:

```json
{
  "name": "calculator",
  "description": "Evaluates simple math expressions like '5 + 3' or '10 * 2'",
  "input_schema": {
    "type": "object",
    "properties": {
      "expression": {
        "type": "string",
        "description": "A math expression like '10 + 5' or '100 / 4'"
      }
    },
    "required": ["expression"]
  }
}
```

**Breaking it down:**
- `"name"`: What Claude calls the tool (must match Python function name)
- `"description"`: Tells Claude when to use this tool
- `"input_schema"`: Defines the input parameters
  - `"properties"`: Lists each parameter (here: just "expression")
  - `"required"`: Which parameters are mandatory (here: "expression" is required)
  - Each parameter has a `"description"` so Claude knows what to put there

### Now Let's Build the Function

We'll create a Python function that takes the expression and calculates it.



In [1]:

# Define the calculator tool schema as a Python dictionary
# This schema tells Claude what the tool does and what inputs it expects
calculator_schema = {
    "name": "calculator",  # Tool name (must be simple, no spaces)
    "description": "Evaluates simple math expressions like '5 + 3' or '10 * 2'",  # Claude reads this to decide when to use it
    "input_schema": {
        "type": "object",  # Input is a JSON object
        "properties": {
            "expression": {  # The calculator expects a parameter called "expression"
                "type": "string",  # The expression should be a text string
                "description": "A math expression like '10 + 5' or '100 / 4' or '50 ** 2'"  # Claude knows to put math here
            }
        },
        "required": ["expression"]  # The "expression" parameter is mandatory
    }
}

# Now let's write the Python function that actually does the calculation
def calculator_tool(expression):  # Takes the math expression as input
    '''
    Execute a simple math calculation.

    Args:
        expression (str): A string like "10 + 5" or "100 / 2"

    Returns:
        str: The result as a formatted string
    '''

    try:  # Start a try/except block to catch errors
        # Use Python's eval() to evaluate the math expression
        # eval() converts a string like "10 + 5" into actual math: 10 + 5 = 15
        # WARNING: eval() can be dangerous with untrusted input, but for a learning
        # session with controlled input, it's fine. In production, use safer alternatives.
        result = eval(expression)  # Evaluate the expression safely (string -> number)

        # Convert the result to a string with nice formatting
        result_string = str(result)  # Convert number to text so we can return it

        # Return the result with a nice message
        return f"Calculator result: {result_string}"  # Return a formatted answer

    except ZeroDivisionError:  # Handle division by zero errors specifically
        # If someone tries "10 / 0", catch it here
        return "Error: Cannot divide by zero"  # Return a friendly error message

    except SyntaxError:  # Handle invalid math expressions
        # If someone tries "10 + + 5" (broken syntax), catch it here
        return f"Error: Invalid expression '{expression}'. Please use valid math syntax."  # Explain the problem

    except Exception as exception_object:  # Catch any other unexpected errors
        # This catches anything else that went wrong
        return f"Error: {str(exception_object)}"  # Return the error message


# Let's test the calculator function with some examples
print("--- Testing Calculator Tool ---")

# Test 1: Simple addition
test_result_1 = calculator_tool("150 + 50")  # Add 150 and 50
print(f"Input: '150 + 50' -> {test_result_1}")  # Show input and result

# Test 2: Multiplication
test_result_2 = calculator_tool("25 * 4")  # Multiply 25 by 4
print(f"Input: '25 * 4' -> {test_result_2}")  # Show input and result

# Test 3: Division
test_result_3 = calculator_tool("100 / 5")  # Divide 100 by 5
print(f"Input: '100 / 5' -> {test_result_3}")  # Show input and result

# Test 4: Complex expression
test_result_4 = calculator_tool("(10 + 5) * 2")  # Parentheses and multiple operations
print(f"Input: '(10 + 5) * 2' -> {test_result_4}")  # Show input and result

# Test 5: Error handling - division by zero
test_result_5 = calculator_tool("50 / 0")  # Try to divide by zero (will error)
print(f"Input: '50 / 0' -> {test_result_5}")  # Show the error message

# Test 6: Error handling - invalid syntax
test_result_6 = calculator_tool("10 + + 5")  # Invalid expression (malformed)
print(f"Input: '10 + + 5' -> {test_result_6}")  # Show the error message

print("\n✅ Calculator tool is working!")  # Print confirmation


--- Testing Calculator Tool ---
Input: '150 + 50' -> Calculator result: 200
Input: '25 * 4' -> Calculator result: 100
Input: '100 / 5' -> Calculator result: 20.0
Input: '(10 + 5) * 2' -> Calculator result: 30
Input: '50 / 0' -> Error: Cannot divide by zero
Input: '10 + + 5' -> Calculator result: 15

✅ Calculator tool is working!



---
## Section 4: Building the Weather Tool

### Weather Tool: Same Structure as Calculator

The weather tool follows the same pattern:
1. **Schema** — Tells Claude about the tool
2. **Function** — Does the actual work

The difference: Instead of math, we'll return mock weather data.

### Why Mock Data?

Real weather APIs require:
- API keys and authentication
- Rate limiting (can't call too often)
- Network requests (slower, sometimes flaky)

**For learning**, we use mock data:
- No API keys needed
- Instant results
- Realistic-looking weather data
- Perfect for understanding tool-calling

In the real world, this function would call a real weather API
(like OpenWeatherMap), but the principle is identical.

### The Weather Tool Schema

```json
{
  "name": "weather_lookup",
  "description": "Returns current weather conditions for a given city",
  "input_schema": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "Name of the city to get weather for (e.g., 'Mumbai', 'Delhi')"
      }
    },
    "required": ["city"]
  }
}
```

**Breaking it down:**
- Takes one input: `"city"` (a string with a city name)
- Returns weather information for that city
- Claude will decide to use this when you ask about weather



In [2]:

# Define the weather lookup tool schema as a Python dictionary
# This schema tells Claude what the tool does and what inputs it expects
weather_schema = {
    "name": "weather_lookup",  # Tool name (matches function name below)
    "description": "Returns current weather conditions for a given city",  # Claude reads this to decide when to use it
    "input_schema": {
        "type": "object",  # Input is a JSON object
        "properties": {
            "city": {  # The weather tool expects a parameter called "city"
                "type": "string",  # The city should be a text string
                "description": "Name of the city to get weather for (e.g., 'Mumbai', 'Delhi', 'Bangalore')"  # Claude knows to put a city name here
            }
        },
        "required": ["city"]  # The "city" parameter is mandatory
    }
}

# Define mock weather data for several Indian cities
# In a real app, this would come from an API like OpenWeatherMap
mock_weather_data = {
    "Mumbai": {  # Data for Mumbai
        "temperature": 28,  # Temperature in Celsius
        "condition": "Sunny",  # Weather condition
        "humidity": 65,  # Humidity percentage
        "wind_speed": 12,  # Wind speed in km/h
        "feels_like": 31  # "Feels like" temperature
    },
    "Delhi": {  # Data for Delhi
        "temperature": 35,
        "condition": "Hot and sunny",
        "humidity": 40,
        "wind_speed": 8,
        "feels_like": 38
    },
    "Bangalore": {  # Data for Bangalore
        "temperature": 26,
        "condition": "Cloudy",
        "humidity": 72,
        "wind_speed": 15,
        "feels_like": 28
    },
    "Chennai": {  # Data for Chennai
        "temperature": 32,
        "condition": "Sunny with humidity",
        "humidity": 75,
        "wind_speed": 18,
        "feels_like": 37
    },
    "Kolkata": {  # Data for Kolkata
        "temperature": 30,
        "condition": "Partly cloudy",
        "humidity": 68,
        "wind_speed": 10,
        "feels_like": 33
    }
}

# Now let's write the Python function that returns weather data
def weather_lookup_tool(city):  # Takes the city name as input
    '''
    Look up weather conditions for a given city (using mock data).

    Args:
        city (str): Name of the city (e.g., "Mumbai", "Delhi")

    Returns:
        str: A formatted weather report
    '''

    # Check if the city name is empty
    if not city or city.strip() == "":  # If city is empty or only whitespace
        return "Error: Please provide a city name"  # Return an error message

    # Capitalize the city name for consistent lookup
    # "mumbai" -> "Mumbai", "DELHI" -> "Delhi"
    city_normalized = city.strip().capitalize()  # Remove whitespace and capitalize first letter

    # Check if we have data for this city
    if city_normalized in mock_weather_data:  # If the city is in our mock data
        # Get the weather data for this city
        weather_info = mock_weather_data[city_normalized]  # Retrieve the dictionary for this city

        # Format the data into a nice string
        weather_report = (  # Build a formatted weather report
            f"Weather in {city_normalized}:\n"  # City name header
            f"  Temperature: {weather_info['temperature']}°C\n"  # Temperature
            f"  Condition: {weather_info['condition']}\n"  # Weather condition
            f"  Humidity: {weather_info['humidity']}%\n"  # Humidity level
            f"  Wind Speed: {weather_info['wind_speed']} km/h\n"  # Wind speed
            f"  Feels Like: {weather_info['feels_like']}°C"  # Feels-like temperature
        )

        # Return the formatted report
        return weather_report  # Return the complete weather information

    else:  # If the city is NOT in our mock data
        # Return a friendly error message
        return f"Error: No weather data available for '{city}'. Available cities: Mumbai, Delhi, Bangalore, Chennai, Kolkata"  # List available cities


# Let's test the weather lookup function with some examples
print("--- Testing Weather Lookup Tool ---\n")

# Test 1: Mumbai weather
test_weather_1 = weather_lookup_tool("Mumbai")  # Look up Mumbai weather
print(f"City: 'Mumbai'\n{test_weather_1}")  # Show the result
print()  # Blank line for readability

# Test 2: Delhi weather
test_weather_2 = weather_lookup_tool("Delhi")  # Look up Delhi weather
print(f"City: 'Delhi'\n{test_weather_2}")  # Show the result
print()

# Test 3: Bangalore weather
test_weather_3 = weather_lookup_tool("Bangalore")  # Look up Bangalore weather
print(f"City: 'Bangalore'\n{test_weather_3}")  # Show the result
print()

# Test 4: Unknown city (error case)
test_weather_4 = weather_lookup_tool("Zzzville")  # Try an unknown city
print(f"City: 'Zzzville'\n{test_weather_4}")  # Show the error message
print()

# Test 5: Empty city name (error case)
test_weather_5 = weather_lookup_tool("")  # Try empty string
print(f"City: '' (empty)\n{test_weather_5}")  # Show the error message
print()

print("✅ Weather lookup tool is working!")  # Print confirmation


--- Testing Weather Lookup Tool ---

City: 'Mumbai'
Weather in Mumbai:
  Temperature: 28°C
  Condition: Sunny
  Humidity: 65%
  Wind Speed: 12 km/h
  Feels Like: 31°C

City: 'Delhi'
Weather in Delhi:
  Temperature: 35°C
  Condition: Hot and sunny
  Humidity: 40%
  Wind Speed: 8 km/h
  Feels Like: 38°C

City: 'Bangalore'
Weather in Bangalore:
  Temperature: 26°C
  Condition: Cloudy
  Humidity: 72%
  Wind Speed: 15 km/h
  Feels Like: 28°C

City: 'Zzzville'
Error: No weather data available for 'Zzzville'. Available cities: Mumbai, Delhi, Bangalore, Chennai, Kolkata

City: '' (empty)
Error: Please provide a city name

✅ Weather lookup tool is working!



### ✅ Checkpoint: Understanding the Weather Tool

**Question 1:** Why do we use mock data instead of a real weather API?

<details>
<summary>Click to reveal answer</summary>

Real weather APIs:
- Require API keys and authentication
- Have rate limits (can't call too often)
- Are slower (network requests)
- Can fail (API downtime)

Mock data:
- Works instantly with no external dependencies
- Perfect for learning and testing
- Always available
- No credentials needed

For understanding tool-calling, mock data is ideal. In production, you'd replace
the mock data dictionary with real API calls.

</details>

**Question 2:** What happens if someone asks for weather in a city that isn't in `mock_weather_data`?

<details>
<summary>Click to reveal answer</summary>

The function checks if the city is in the dictionary:
```python
if city_normalized in mock_weather_data:
    # Return weather data
else:
    # Return error message with list of available cities
```

If the city isn't found, it returns a helpful error message listing which cities
are available (Mumbai, Delhi, Bangalore, Chennai, Kolkata).

This is better than crashing or returning misleading data.

</details>

**Question 3:** Why do we normalize the city name with `.capitalize()`?

<details>
<summary>Click to reveal answer</summary>

The user might type the city in different ways:
- "mumbai" (lowercase)
- "MUMBAI" (uppercase)
- "Mumbai" (correct capitalization)

Without normalization, only "Mumbai" would match. With `.capitalize()`:
- "mumbai" -> "Mumbai" ✓
- "MUMBAI" -> "Mumbai" ✓
- "Mumbai" -> "Mumbai" ✓

This makes the tool more user-friendly.

</details>



---
## Section 5: Watching Claude Choose & Live Exercises

### The Complete Tool-Calling Loop

Now we'll see Claude actually using our tools. Here's what happens:

```
Your Prompt
    ↓
Claude reads prompt + sees our 2 tool definitions
    ↓
Claude decides: "Do I need calculator? Weather? Both? Neither?"
    ↓
Claude returns tool_use blocks (if any) with requested inputs
    ↓
Your code sees tool_use blocks and executes the functions
    ↓
Your code sends results back to Claude
    ↓
Claude reads results and gives final answer
```

Let's build the code to make this work!



In [3]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 3.1 MB/s eta 0:00:00


In [4]:
from anthropic import Anthropic
from google.colab import userdata
key = userdata.get('MY_API_KEY')

client = Anthropic(api_key=key)

In [5]:

import json  # Import json to work with structured tool data

# Create a tools list that we'll send to Claude
# This list contains both tools: calculator and weather
tools_list = [
    calculator_schema,  # Include the calculator tool schema
    weather_schema  # Include the weather tool schema
]

# Function to process tool calls (execute the requested tools)
def process_tool_calls(tool_calls_list, response_content):  # Takes list of tool calls and full response
    '''
    Execute tool calls requested by Claude.

    Args:
        tool_calls_list (list): List of tool_use blocks from Claude
        response_content (list): Full response content from Claude

    Returns:
        list: Tool results to send back to Claude
    '''

    tool_results_list = []  # Start with empty list of results

    # Loop through each tool call Claude requested
    for tool_use_block in tool_calls_list:  # For each tool request

        # Extract the tool name Claude requested
        tool_name = tool_use_block.name  # e.g., "calculator" or "weather_lookup"

        # Extract the inputs Claude specified for this tool
        tool_input = tool_use_block.input  # Dictionary with the input parameters

        # Extract the unique ID for this tool call (needed to match with result)
        tool_use_id = tool_use_block.id  # Unique identifier for this tool call

        print(f"--- Tool Request Detected ---")  # Separator for readability
        print(f"Tool Name: {tool_name}")  # Show which tool
        print(f"Tool Input: {tool_input}")  # Show the inputs

        # Decide which tool to run based on the tool name
        if tool_name == "calculator":  # If Claude requested the calculator
            # Extract the expression input
            expression = tool_input["expression"]  # Get the math expression

            # Run the calculator function
            result = calculator_tool(expression)  # Execute the tool

            print(f"Result: {result}")  # Show the result

        elif tool_name == "weather_lookup":  # If Claude requested the weather tool
            # Extract the city input
            city = tool_input["city"]  # Get the city name

            # Run the weather function
            result = weather_lookup_tool(city)  # Execute the tool

            print(f"Result: {result}")  # Show the result

        else:  # If Claude requested a tool we don't recognize
            # This shouldn't happen, but we handle it anyway
            result = f"Error: Unknown tool '{tool_name}'"  # Return an error

        # Create a tool result object to send back to Claude
        tool_result = {
            "type": "tool_result",  # Marks this as a tool result (not a user message)
            "tool_use_id": tool_use_id,  # Match this result to the original request
            "content": result  # The actual result from the tool
        }

        # Add this result to our list
        tool_results_list.append(tool_result)  # Accumulate all results

    # Return all tool results
    return tool_results_list  # Return the complete list of results


# Function to extract tool_use blocks from Claude's response
def extract_tool_uses(response_object):  # Takes Claude's response
    r'''
    Find all tool_use blocks in Claude response.

    Args:
        response_object: The response from Claude API

    Returns:
        list: List of tool_use content blocks (empty if none)
    '''

    tool_use_blocks = []  # Start with empty list

    # Loop through each content block in the response
    for content_block in response_object.content:  # For each piece of the response

        # Check if this block is a tool_use request
        if content_block.type == "tool_use":  # If it's a tool request
            # Add it to our list
            tool_use_blocks.append(content_block)  # Save this tool request

    # Return all tool_use blocks found
    return tool_use_blocks  # Return the tool requests


# Function to run a complete tool-calling interaction
def run_tool_calling_interaction(user_prompt):  # Takes the user's question
    '''
    Send a prompt to Claude, process tool calls, and get the final answer.

    Args:
        user_prompt (str): The user question
    '''

    print(f"\n{'='*60}")  # Print separator
    print(f"USER PROMPT: {user_prompt}")  # Show the user's question
    print(f"{'='*60}")  # Print separator

    # Step 1: Send the prompt to Claude with our tools
    print("\n[Step 1] Sending prompt to Claude with tool definitions...")  # Status message

    # Create the message request to Claude
    initial_response = client.messages.create(
        model="claude-haiku-4-5-20251001",  # Use Claude Sonnet model
        max_tokens=1000,  # Allow up to 1000 tokens in response
        tools=tools_list,  # Provide our tools to Claude
        messages=[
            {
                "role": "user",  # This is from the user
                "content": user_prompt  # The user's question
            }
        ]
    )

    # Step 2: Check if Claude requested any tools
    print("\n[Step 2] Checking Claude's response for tool requests...")  # Status message

    # Extract any tool_use blocks from Claude's response
    detected_tool_uses = extract_tool_uses(initial_response)  # Find tool requests

    # Show how many tools Claude requested
    tool_count = len(detected_tool_uses)  # Count the requests
    print(f"Claude requested {tool_count} tool(s)")  # Display count

    # If Claude didn't request any tools, show the answer directly
    if tool_count == 0:  # If no tools were requested
        print("\n[No tools needed] Claude answered directly:")  # Status message

        # Extract and show Claude's text response
        for content_block in initial_response.content:  # For each content piece
            if content_block.type == "text":  # If it's text (not tool_use)
                print(f"\nCLAUDE'S ANSWER:\n{content_block.text}")  # Show the answer

        return  # Exit the function (we're done)

    # Step 3: Execute the tools Claude requested
    print(f"\n[Step 3] Executing {tool_count} tool(s)...")  # Status message

    # Run all the tools Claude requested
    tool_results = process_tool_calls(detected_tool_uses, initial_response)  # Execute tools

    # Step 4: Send results back to Claude
    print("\n[Step 4] Sending tool results back to Claude...")  # Status message

    # Build the messages list for the follow-up request
    # Include the original prompt and Claude's response with tool requests
    messages_with_results = [
        {
            "role": "user",  # Original user message
            "content": user_prompt  # The original prompt
        },
        {
            "role": "assistant",  # Claude's response
            "content": initial_response.content  # All of Claude's content (text + tool requests)
        },
        {
            "role": "user",  # Now the user provides tool results
            "content": tool_results  # The results from executing tools
        }
    ]

    # Send the results back to Claude and get the final answer
    final_response = client.messages.create(
        model="claude-haiku-4-5-20251001",  # Same model
        max_tokens=1000,  # Allow up to 1000 tokens
        tools=tools_list,  # Still provide tools (in case Claude needs them again)
        messages=messages_with_results  # Messages including tool results
    )

    # Step 5: Display Claude's final answer
    print("\n[Step 5] Claude's final answer:")  # Status message

    # Extract and display Claude's final text response
    for content_block in final_response.content:  # For each content piece
        if content_block.type == "text":  # If it's text
            print(f"\nCLAUDE'S FINAL ANSWER:\n{content_block.text}")  # Show the answer


print("✅ Tool-calling loop is set up and ready!")  # Confirmation


✅ Tool-calling loop is set up and ready!



### Example 1: Calculator Only

Let's ask Claude a math question:


In [6]:

# Example 1: Ask Claude to do math (calculator only)
prompt_1 = "What is 1500 divided by 30?"  # Simple math question

# Run the interaction
run_tool_calling_interaction(prompt_1)  # Send to Claude



USER PROMPT: What is 1500 divided by 30?

[Step 1] Sending prompt to Claude with tool definitions...

[Step 2] Checking Claude's response for tool requests...
Claude requested 1 tool(s)

[Step 3] Executing 1 tool(s)...
--- Tool Request Detected ---
Tool Name: calculator
Tool Input: {'expression': '1500 / 30'}
Result: Calculator result: 50.0

[Step 4] Sending tool results back to Claude...

[Step 5] Claude's final answer:

CLAUDE'S FINAL ANSWER:
1500 divided by 30 is **50**.



### Example 2: Weather Only

Let's ask Claude about the weather:


In [7]:

# Example 2: Ask Claude about weather (weather tool only)
prompt_2 = "Is it hot in Delhi right now? How does it compare to Bangalore?"  # Weather question

# Run the interaction
run_tool_calling_interaction(prompt_2)  # Send to Claude



USER PROMPT: Is it hot in Delhi right now? How does it compare to Bangalore?

[Step 1] Sending prompt to Claude with tool definitions...

[Step 2] Checking Claude's response for tool requests...
Claude requested 2 tool(s)

[Step 3] Executing 2 tool(s)...
--- Tool Request Detected ---
Tool Name: weather_lookup
Tool Input: {'city': 'Delhi'}
Result: Weather in Delhi:
  Temperature: 35°C
  Condition: Hot and sunny
  Humidity: 40%
  Wind Speed: 8 km/h
  Feels Like: 38°C
--- Tool Request Detected ---
Tool Name: weather_lookup
Tool Input: {'city': 'Bangalore'}
Result: Weather in Bangalore:
  Temperature: 26°C
  Condition: Cloudy
  Humidity: 72%
  Wind Speed: 15 km/h
  Feels Like: 28°C

[Step 4] Sending tool results back to Claude...

[Step 5] Claude's final answer:

CLAUDE'S FINAL ANSWER:
Yes, it's quite hot in Delhi right now! Here's the comparison:

**Delhi:**
- Temperature: 35°C (feels like 38°C)
- Condition: Hot and sunny
- Humidity: 40%
- Wind Speed: 8 km/h

**Bangalore:**
- Temperature


### Example 3: Both Tools Together

Let's ask Claude to use both tools in one prompt:


In [8]:

# Example 3: Ask Claude for math AND weather (both tools)
prompt_3 = (
    "Calculate 350 + 200. "  # Math part
    "Also tell me the weather in Bangalore. "  # Weather part
    "If the weather in Bangalore is good, would it be a nice day to travel from Mumbai? "  # Reasoning part
)

# Run the interaction
run_tool_calling_interaction(prompt_3)  # Send to Claude



USER PROMPT: Calculate 350 + 200. Also tell me the weather in Bangalore. If the weather in Bangalore is good, would it be a nice day to travel from Mumbai? 

[Step 1] Sending prompt to Claude with tool definitions...

[Step 2] Checking Claude's response for tool requests...
Claude requested 3 tool(s)

[Step 3] Executing 3 tool(s)...
--- Tool Request Detected ---
Tool Name: calculator
Tool Input: {'expression': '350 + 200'}
Result: Calculator result: 550
--- Tool Request Detected ---
Tool Name: weather_lookup
Tool Input: {'city': 'Bangalore'}
Result: Weather in Bangalore:
  Temperature: 26°C
  Condition: Cloudy
  Humidity: 72%
  Wind Speed: 15 km/h
  Feels Like: 28°C
--- Tool Request Detected ---
Tool Name: weather_lookup
Tool Input: {'city': 'Mumbai'}
Result: Weather in Mumbai:
  Temperature: 28°C
  Condition: Sunny
  Humidity: 65%
  Wind Speed: 12 km/h
  Feels Like: 31°C

[Step 4] Sending tool results back to Claude...

[Step 5] Claude's final answer:

CLAUDE'S FINAL ANSWER:
Great! H


---

## Congratulations! You've Completed the Course

### What You Learned Today

✅ **Understanding Tools**
- Tools are pairs of schema + function
- Schema tells Claude about a tool; function does the work
- Claude automatically chooses the right tool(s) for each question

✅ **How Tool Selection Works**
- Claude reads the user's prompt + tool definitions
- Claude decides: "Do I need tools? Which ones?"
- Claude sends back a tool_use request (not the answer)
- Your code executes the tools and returns results
- Claude reads results and gives the final answer

✅ **Building Tools**
- A tool has a JSON schema (name, description, input_schema)
- A tool has a Python function that does the actual work
- The schema and function must match (same name, compatible inputs)
- Always include error handling in your functions

✅ **The Tool-Calling Loop**
```
User Prompt -> Claude Reads + Decides -> Claude Requests Tools -> Your Code Executes
    -> Results Back to Claude -> Claude Answers
```

✅ **Practical Skills**
- Building calculator and weather tools
- Creating tool schemas that Claude understands
- Executing tool calls based on Claude's requests
- Handling multiple tools in one response
- Error handling for edge cases

### Next Steps: What To Do After This Course

1. **Try More Tools:** Design tools for currency, unit conversion, data lookup, etc.

2. **Use Real APIs:** Replace mock data with real APIs
   - Weather: OpenWeatherMap, Weather API
   - Currency: Forex API, Exchange Rate API
   - News: NewsAPI, NewsData.io
   - Math: WolframAlpha API

3. **Build Agents:** Create tools that work together:
   - A research agent (web search + summarization)
   - A travel planner (weather + flights + hotels)
   - A personal assistant (calendar + email + notes)

4. **Read the Docs:**
   - Anthropic Tool Use: https://docs.anthropic.com/en/docs/tool-use
   - API Reference: https://docs.anthropic.com/en/api/messages

### Key Resources

- **Anthropic Documentation**: https://docs.anthropic.com
- **Claude API Reference**: https://docs.anthropic.com/en/api/messages
- **Tool Use Guide**: https://docs.anthropic.com/en/docs/tool-use
- **Python SDK**: https://github.com/anthropics/anthropic-sdk-python

### Feedback & Questions

If you have questions:
1. Check the Anthropic documentation
2. Review the tool schemas and functions in this notebook
3. Experiment! Break things and fix them to learn

### One More Thing: API Keys

Remember:
- Never commit API keys to GitHub
- Never share keys in code snippets
- Always use environment variables in production
- Rotate keys regularly
- Set spending limits on your API key

### Thank You!

You now understand how Claude's tool-calling works at a deep level.
Go build something amazing!

---


## Additional Resources & Links

### Official Documentation
- **Tool Use Docs**: https://docs.anthropic.com/en/docs/tool-use
- **Messages API**: https://docs.anthropic.com/en/api/messages
- **Python SDK**: https://github.com/anthropics/anthropic-sdk-python

### Learn More About
- **Function Calling in LLMs**: https://en.wikipedia.org/wiki/Function_(chemistry) (general concept)
- **JSON Schema**: https://json-schema.org/ (schema validation standard)
- **REST APIs**: https://www.rest.org/ (web service basics)

### Fun Challenges

1. **Build a math quiz tool** that generates random math problems
2. **Create a language translator tool** with mock translations
3. **Design a movie recommendation tool** with a hardcoded database
4. **Make a poem generator tool** that creates rhyming verses
5. **Build a contact manager** that stores and retrieves contacts

### Common Mistakes to Avoid

- Never forget the `"required"` field in schemas
- Never return dictionaries instead of strings from tool functions
- Never use wrong parameter names (schema vs. function mismatch)
- Never skip error handling in tool functions
- Never share API keys in code
- Never call tools synchronously when they could be async
- Never skip input validation in tool functions

### Tips for Success

- Always test tool functions independently first
- Use clear, descriptive names for tools and parameters
- Include helpful error messages in your tools
- Validate all inputs (empty strings, negative numbers, etc.)
- Keep tool descriptions concise but clear
- Use type hints in Python for clarity
- Comment every line of code (like we did!)

### Glossary

- **Tool**: A pair of schema + function that Claude can use
- **Schema**: JSON definition of a tool (name, description, inputs)
- **Tool Use**: Claude's request to execute a tool
- **Tool Result**: The output from executing a tool
- **Mock Data**: Fake data used for testing (not from a real API)
- **API Key**: Secret credential to authenticate with an API
- **Parameter**: Input that a tool expects
- **Token**: Unit of text that Claude's API counts for billing

---

**Good luck with your AI journey!**
